In [ ]:
# ===============================================
# DRKG pipeline: robust, CPU/GPU safe
# - GCN, GAT, GraphSAGE, GIN + DistMult baseline
# - NeighborLoader (with ClusterLoader fallback)
# - Inductive disease-level evaluation
# - Mechanistic path regularization (C->G->D)
# - Ensembles + MC Dropout
# - Ranked CSV with supporting genes
# ===============================================

import os, json, random
from collections import defaultdict
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score

# -------------------- CONFIG --------------------
GRAPH_DIR =r"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph"
OUT_DIR = os.path.join(GRAPH_DIR, "pipeline_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# Hyperparameters
HDIM = 128
OUTDIM = 128
EPOCHS = 6
BATCH_SIZE = 2048
NEIGHBOR_SAMPLES = [20, 10]
LR = 1e-3
WEIGHT_DECAY = 1e-6
PATH_REG_WEIGHT = 0.5
NEG_RATIO = 1
ENSEMBLE_SIZE = 2
MC_RUNS = 30
TOP_K = 50

# -------------------- Load files --------------------
def safe_read_lines(path):
    if not os.path.exists(path): return []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return [ln.rstrip() for ln in f]

edge_index = torch.load(os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/edge_index.pt")).long()
edge_type  = torch.load(os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/edge_type.pt")).long()
entities_lines = safe_read_lines(os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/entities.txt"))
relation_lines = safe_read_lines(os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/relations.txt"))

def read_csv_triples(path):
    if not os.path.exists(path): return []
    df = pd.read_csv(path, header=None, dtype=str)
    # skip header if present
    if df.shape[1] >= 3 and list(df.iloc[0].str.lower())[:3] == ['head','relation','tail']:
        df = df.iloc[1:].reset_index(drop=True)
    return [(str(r[0]).strip(), str(r[1]).strip(), str(r[2]).strip()) for r in df.values if len(r)>=3]

train_triples = read_csv_triples(os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/train_inductive.csv"))
val_triples   = read_csv_triples(os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/val_inductive.csv"))
test_triples  = read_csv_triples(os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/test_inductive.csv"))

print("Loaded:", len(entities_lines), "entities,", len(relation_lines), "relations")
print("Edge index shape:", tuple(edge_index.shape))
print("Train/Val/Test triples:", len(train_triples), len(val_triples), len(test_triples))

# -------------------- Parse entities --------------------
ent2name = {}
ent2type = {}
for ln in entities_lines:
    parts = ln.split("\t")
    if len(parts) >= 2:
        gid_str, label = parts[0].strip(), parts[1].strip()
        ent2name[gid_str] = label
        lab_low = label.lower()
        if any(x in lab_low for x in ["compound","drug","chembl","atc"]):
            typ = "compound"
        elif any(x in lab_low for x in ["gene","hgnc","entrez"]):
            typ = "gene"
        elif any(x in lab_low for x in ["disease","doid","mesh"]):
            typ = "disease"
        else:
            typ = "other"
        ent2type[gid_str] = typ

# mapping: label -> gid
label_to_gid = {v:k for k,v in ent2name.items()}

# int gid mapping
gidstr_to_int = {k:int(k) for k in ent2name if k.isdigit()}

num_nodes = int(edge_index.max().item()) + 1
all_compound_gids = [int(k) for k,v in ent2type.items() if v=="compound"]
all_disease_gids  = [int(k) for k,v in ent2type.items() if v=="disease"]

# -------------------- Parse relations --------------------
rid_to_relname = {}
for ln in relation_lines:
    parts = ln.split("\t")
    if len(parts)>=2: rid_to_relname[parts[0].strip()] = parts[1].strip()

def relname_indicates(relname, patterns):
    if relname is None: return False
    rl = str(relname).lower()
    return any(p in rl for p in patterns)

pattern_cg = ["compound","target","bind","cbg","dgidb","gnbr::a","gnbr::b","gnbr::e"]
pattern_gd = ["gene","assoc","associated","gnbr::d","gnbr::g","gnbr::md","gnbr::mp","gnbr::j"]

# -------------------- Map triple labels -> int gids --------------------
def map_triple_label_to_int(triple):
    h,r,t = triple
    if h in label_to_gid and t in label_to_gid:
        try: return int(label_to_gid[h]), r, int(label_to_gid[t])
        except: return None
    try:
        hi, ti = int(h), int(t)
        if 0<=hi<num_nodes and 0<=ti<num_nodes: return hi,r,ti
    except: pass
    # fallback: match suffix
    matched_h = matched_t = None
    for label,gidstr in label_to_gid.items():
        if label.endswith(h): matched_h=int(gidstr)
        if label.endswith(t): matched_t=int(gidstr)
    if matched_h is not None and matched_t is not None: return matched_h,r,matched_t
    return None

train_int = [map_triple_label_to_int(t) for t in train_triples]; train_int=[t for t in train_int if t]
val_int   = [map_triple_label_to_int(t) for t in val_triples]; val_int=[t for t in val_int if t]
test_int  = [map_triple_label_to_int(t) for t in test_triples]; test_int=[t for t in test_int if t]

# -------------------- Mechanistic pairs --------------------
comp_gene_pairs = set(); gene_disease_pairs = set(); comp_disease_pairs = set()
for h,r,t in train_int:
    h_type, t_type = ent2type.get(str(h),"other"), ent2type.get(str(t),"other")
    rname = rid_to_relname.get(str(r), str(r))
    if h_type=="compound" and t_type=="gene" or relname_indicates(rname, pattern_cg): comp_gene_pairs.add((h,t))
    if h_type=="gene" and t_type=="disease" or relname_indicates(rname, pattern_gd): gene_disease_pairs.add((h,t))
    if h_type=="compound" and t_type=="disease" or "treat" in str(rname).lower() or "indicat" in str(rname).lower(): comp_disease_pairs.add((h,t))

print("Mechanistic pairs (train) C->G:", len(comp_gene_pairs),
      "G->D:", len(gene_disease_pairs), "C->D:", len(comp_disease_pairs))

# -------------------- Prepare labeled compound-disease pairs --------------------
def build_cd_pairs_from_triples(triples):
    pos = []
    for h,r,t in triples:
        h_type,t_type=ent2type.get(str(h),"other"),ent2type.get(str(t),"other")
        if (h_type=="compound" and t_type=="disease") or "treat" in str(r).lower() or "indicat" in str(r).lower(): pos.append((h,t))
    return list(set(pos))

train_pos_cd = build_cd_pairs_from_triples(train_int)
val_pos_cd   = build_cd_pairs_from_triples(val_int)
test_pos_cd  = build_cd_pairs_from_triples(test_int)

def negative_sample_for_list(comp_list, neg_ratio=1):
    negs=[]
    for c in comp_list:
        for _ in range(neg_ratio):
            negs.append((c, random.choice(all_disease_gids)))
    return negs

train_pairs = [(c,d,1) for c,d in train_pos_cd] + [(c,d,0) for c,d in negative_sample_for_list([c for c,_ in train_pos_cd],NEG_RATIO)]
val_pairs   = [(c,d,1) for c,d in val_pos_cd] + [(c,d,0) for c,d in negative_sample_for_list([c for c,_ in val_pos_cd],1)]
test_pairs  = [(c,d,1) for c,d in test_pos_cd] + [(c,d,0) for c,d in negative_sample_for_list([c for c,_ in test_pos_cd],1)]

random.shuffle(train_pairs)
print("Train/Val/Test pairs:", len(train_pairs), len(val_pairs), len(test_pairs))

# -------------------- CPU-safe model base --------------------
class BaseGNN(nn.Module):
    def __init__(self, num_nodes, hidden_dim=HDIM, outdim=OUTDIM, dropout=0.3):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, hidden_dim)
        self.dropout = dropout
    def embed(self): return self.node_emb.weight

from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv

class GCNModel(BaseGNN):
    def __init__(self,num_nodes,hidden_dim=HDIM,outdim=OUTDIM,dropout=0.3):
        super().__init__(num_nodes,hidden_dim,outdim,dropout)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, outdim)
    def forward(self, edge_index, edge_type=None):
        x = self.node_emb.weight
        x = self.conv1(x, edge_index)
        x = F.relu(x); x=F.dropout(x,p=self.dropout,training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GATModel(BaseGNN):
    def __init__(self,num_nodes,hidden_dim=HDIM,outdim=OUTDIM,heads=2,dropout=0.3):
        super().__init__(num_nodes,hidden_dim,outdim,dropout)
        self.conv1 = GATConv(hidden_dim, hidden_dim//heads, heads=heads)
        self.conv2 = GATConv(hidden_dim, outdim, heads=1)
    def forward(self, edge_index, edge_type=None):
        x = self.node_emb.weight
        x = self.conv1(x, edge_index)
        x = F.elu(x); x=F.dropout(x,p=self.dropout,training=self.training)
        x = self.conv2(x, edge_index)
        return x

class SAGEModel(BaseGNN):
    def __init__(self,num_nodes,hidden_dim=HDIM,outdim=OUTDIM,dropout=0.3):
        super().__init__(num_nodes,hidden_dim,outdim,dropout)
        self.conv1 = SAGEConv(hidden_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, outdim)
    def forward(self, edge_index, edge_type=None):
        x = self.node_emb.weight
        x = self.conv1(x, edge_index)
        x = F.relu(x); x=F.dropout(x,p=self.dropout,training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GINModel(BaseGNN):
    def __init__(self,num_nodes,hidden_dim=HDIM,outdim=OUTDIM,dropout=0.3):
        super().__init__(num_nodes,hidden_dim,outdim,dropout)
        nn1 = nn.Sequential(nn.Linear(hidden_dim,hidden_dim), nn.ReLU(), nn.Linear(hidden_dim,hidden_dim))
        self.conv1 = GINConv(nn1)
        nn2 = nn.Sequential(nn.Linear(hidden_dim,hidden_dim), nn.ReLU(), nn.Linear(hidden_dim,outdim))
        self.conv2 = GINConv(nn2)
    def forward(self, edge_index, edge_type=None):
        x = self.node_emb.weight
        x = self.conv1(x, edge_index)
        x = F.relu(x); x=F.dropout(x,p=self.dropout,training=self.training)
        x = self.conv2(x, edge_index)
        return x

# -------------------- Scoring --------------------
def score_pairs(node_embs, pairs):
    X = torch.stack([node_embs[c] * node_embs[d] for c,d,_ in pairs])
    scores = torch.sum(X, dim=1)
    return torch.sigmoid(scores)

# -------------------- Training function --------------------
def train_model(model, pairs, edge_index, lr=LR, epochs=EPOCHS, path_reg=False, comp_gene_pairs=None, gene_disease_pairs=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCELoss()
    model.train()
    for ep in range(epochs):
        total_loss=0
        random.shuffle(pairs)
        for i in range(0,len(pairs),BATCH_SIZE):
            batch = pairs[i:i+BATCH_SIZE]
            batch_c = torch.tensor([c for c,d,l in batch],dtype=torch.long,device=DEVICE)
            batch_d = torch.tensor([d for c,d,l in batch],dtype=torch.long,device=DEVICE)
            batch_y = torch.tensor([l for c,d,l in batch],dtype=torch.float,device=DEVICE)
            optimizer.zero_grad()
            embs = model(edge_index.to(DEVICE))
            pred = torch.sum(embs[batch_c]*embs[batch_d],dim=1).sigmoid()
            loss = criterion(pred,batch_y)
            if path_reg:
                # Mechanistic path: C->G->D
                reg_loss = 0
                for c,d in [(c,d) for c,d,l in batch if l==1]:
                    # sample a gene from comp_gene_pairs & gene_disease_pairs
                    genes = [g for (cg,g) in comp_gene_pairs if cg==c]
                    genes2 = [g for (g2,d2) in gene_disease_pairs if d2==d and g2 in genes]
                    for g in genes2:
                        reg_loss += ((embs[c] + embs[g] - embs[d])**2).sum()
                loss += PATH_REG_WEIGHT*reg_loss/ (len(batch)+1)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {ep+1}/{epochs} Loss: {total_loss/len(pairs):.6f}")

# -------------------- MC Dropout / Ensemble prediction --------------------
def predict_with_uncertainty(model, edge_index, pairs, runs=MC_RUNS):
    model.train()  # enable dropout
    all_preds=[]
    for _ in range(runs):
        embs = model(edge_index.to(DEVICE))
        preds = score_pairs(embs, pairs).detach().cpu().numpy()
        all_preds.append(preds)
    mean_preds = np.mean(all_preds, axis=0)
    std_preds  = np.std(all_preds, axis=0)
    return mean_preds, std_preds

# -------------------- Ranked CSV --------------------
def save_ranked_csv(pairs, mean_preds, std_preds, outpath):
    df = pd.DataFrame(pairs, columns=["Compound","Disease","Label"])
    df["Pred"] = mean_preds
    df["Std"]  = std_preds
    df["Rank"] = df["Pred"].rank(method="min",ascending=False)
    df.sort_values("Rank",inplace=True)
    df.to_csv(outpath,index=False)
    print("Saved ranked CSV:", outpath)

# -------------------- Example run --------------------
model = GCNModel(num_nodes).to(DEVICE)
train_model(model, train_pairs, edge_index, path_reg=True, comp_gene_pairs=comp_gene_pairs, gene_disease_pairs=gene_disease_pairs)
mean_preds,std_preds = predict_with_uncertainty(model, edge_index, test_pairs)
save_ranked_csv(test_pairs, mean_preds, std_preds, os.path.join(OUT_DIR,"ranked_test.csv"))


Loaded: 94046 entities, 107 relations
Edge index shape: (2, 5261827)
Train/Val/Test triples: 5261827 584648 27786
Mechanistic pairs (train) C->G: 1901509 G->D: 3060674 C->D: 60568
Train/Val/Test pairs: 121136 13090 3336


NameError: free variable 'g' referenced before assignment in enclosing scope